# Fitting

In [33]:
import sys
sys.path.insert(0, '../../src/')

import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import tensorflow as tf

from qiskit.quantum_info import Operator
from tqdm.notebook import tqdm

from kraus_channels import KrausMap, isomery_to_kraus
from loss_functions import ProbabilityMSE, ProbabilityRValue
from optimization import ModelSPAM, ModelQuantumMap, Logger, model_saver
from quantum_channel import channel_fidelity
from experimental import counts_to_probs, generate_pauliInput_circuits, generate_pauli_circuits, marginalize_counts
from spam import SPAM, InitialState, CorruptionMatrix
from utils import saver
from quantum_circuits import pqc_basic
from spectrum import channel_spectrum, complex_spacing_ratio
from quantum_circuits import integrable_circuit, nonintegrable_circuit


#np.set_printoptions(threshold=sys.maxsize)
np.set_printoptions(precision=4)

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

In [18]:
def load_data(filename, n, seed, L):
    with open(filename, 'rb') as f:
        data = pkl.load(f)


    data = marginalize_counts(data, 0)

    targets = counts_to_probs(data)
    targets_spam = targets[:6**n]
    targets_map = targets[6**n:]

    np.random.seed(seed)
    
    circuit_target = integrable_circuit(n+1, L)
    unitary = Operator(circuit_target).data

    inputs_spam, _ = generate_pauliInput_circuits(n)
        
    inputs_map, circuit_list_map = (
                generate_pauli_circuits(n, None, N=5000-6**n)
        )
    
    return inputs_spam, targets_spam, inputs_map, targets_map, unitary

def fit_spam(inputs, 
             targets,
             num_iter = 3000,
             verbose = False):
    d = targets.shape[1]
    spam_model = SPAM(init = InitialState(d),
                    povm = CorruptionMatrix(d),
                    #povm = POVM(d),
                    )

    spam_opt = ModelSPAM(spam_model, tf.keras.optimizers.Adam(learning_rate=0.01))
        
    spam_opt.pretrain(100, verbose=False)

    spam_opt.train(inputs = inputs,
                    targets = targets,
                    num_iter = num_iter,
                    verbose = verbose,
                )
    
    return spam_model
    

def fit_model(inputs, 
              targets, 
              spam_model,
              num_iter = 3000,
              verbose=False):
    d = targets.shape[1]
    model = ModelQuantumMap(channel = KrausMap(d = d, 
                                        rank = d**2,
                                        spam = spam_model,
                                        ),
                    loss_function = ProbabilityMSE(),
                    optimizer = tf.optimizers.Adam(learning_rate=0.01),
                    logger = Logger(loss_function_list = [ProbabilityRValue()], sample_freq=100),
                )

    model.train(inputs = inputs,
                targets = targets,
                inputs_val = [inputs],
                targets_val = [targets],
                num_iter = num_iter,
                N = 500,
                verbose=verbose
                )
    
    return model

In [10]:
path = 'data/chaos_exp_data_20251022/baseline_L=5_20251019/'
n = 4
d = 2**n
L = 5

spam_list = []
model_list = []

for i in tqdm(range(5)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)



model_saver(spam_list, f'models/integrable_spam_{n}_L=5.model')
model_saver(model_list, f'models/integrable_baseline_model_{n}_L=5.model')

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

2025-10-22 19:52:55.785543: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.
2025-10-22 19:52:56.040398: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.


[-0.0005101941603138371]


2025-10-22 19:54:01.459227: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.
2025-10-22 19:54:01.731777: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.


...

[0.0030624447413927225]
[0.5327910410720917]
[0.928490325036901]
[0.990296969759848]
[0.9952840853179982]
[0.996319760372369]
[0.9968084275130608]
[0.9970960090575356]
[0.997289413962075]
[0.9974208434699814]
[0.9975175663253002]
[0.9975980026528146]
[0.9976544060547492]
[0.9977053151472115]
[0.997737982807565]
[0.9977748163814709]
[0.9978020245984521]
[0.9978199927811086]
[0.9978383306872233]
[0.9978634541563031]
[0.9978714449268854]
[0.9978851886603874]
[0.9978956737756003]
[0.9979056690301416]
[0.9979180784033268]
[0.9979131987766666]
[0.9979227315887402]
[0.9979204485412372]
[0.9979358915307087]
[0.997934648220857]
[0.9979445416599241]


  0%|          | 0/3000 [00:00<?, ?it/s]

[0.0014157692829536161]
[0.5362666594417675]
[0.9192675616673127]
[0.9889597547988391]
[0.9953072824137134]
[0.996570204663]
[0.997126723712056]
[0.9974493371986902]
[0.9976616065617249]
[0.9978032514659015]
[0.997908951060807]
[0.9979884666168903]
[0.998047944701347]
[0.9980994563901913]
[0.9981399487102876]
[0.998173597501239]
[0.9982012291106174]
[0.9982230702468721]
[0.9982408483569688]
[0.9982595057053628]
[0.9982707935590103]
[0.9982851356974025]
[0.9982927948918517]
[0.998301296252305]
[0.9983081519629711]
[0.9983208847355342]
[0.9983187917264162]
[0.9983299123565322]
[0.9983300892489825]
[0.9983387404552981]
[0.9983420660993185]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.0021347966835600918]
[0.5043978530584199]
[0.9211276105270608]
[0.9896879204322097]
[0.9952158265765183]
[0.9963180395441996]
[0.9968126679758581]
[0.9971083929752683]
[0.9973059539503011]
[0.9974451024363167]
[0.9975539946550958]
[0.9976340848286741]
[0.9976961821716389]
[0.9977481975855239]
[0.9977935903306534]
[0.9978300216676066]
[0.9978578460151638]
[0.9978820544548913]
[0.9979046953538702]
[0.9979193945206662]
[0.9979359378309209]
[0.9979494419756018]
[0.997962833950307]
[0.9979758867692597]
[0.997969474940034]
[0.9979811276324888]
[0.997997433384745]
[0.9979935192857525]
[0.9980032339692249]
[0.9980083342912169]
[0.9980111589776194]


In [25]:
path = 'data/chaos_exp_data_20251022/baseline_L=20_20251019/'
n = 4
d = 2**n
L = 20

spam_list = []
model_list = []

for i in tqdm(range(5)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)



model_saver(spam_list, f'models/integrable_spam_{n}_L=20.model')
model_saver(model_list, f'models/integrable_baseline_model_{n}_L=20.model')

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.005259900888176849]
[0.6650159623475176]
[0.9529116563538734]
[0.9788541531467106]
[0.9855499653115776]
[0.9886265905289767]
[0.9903579497596607]
[0.9914425149974282]
[0.9921757534156355]
[0.9926862915355279]
[0.9930676208830581]
[0.9933655258245683]
[0.9935838332712854]
[0.9937584568165357]
[0.9939056629944636]
[0.9940206066183332]
[0.9941294342946773]
[0.9942056472785594]
[0.9942723535302608]
[0.9943350803680826]
[0.9943837971303724]
[0.9944247010864056]
[0.9944576457633381]
[0.994486630161054]
[0.9945201939296913]
[0.9945402153619233]
[0.9945507025362718]
[0.9945760608811935]
[0.9945892481745331]
[0.9946161781233932]
[0.9946143664799462]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.003283016953712581]
[0.679642525831476]
[0.9585760470643273]
[0.9821324061129172]
[0.9879409633631638]
[0.9905697434337435]
[0.9920449637088282]
[0.9929546910884526]
[0.9935624755531665]
[0.9939941725408548]
[0.9943058246662837]
[0.9945534731120484]
[0.9947413802190717]
[0.9948897395733902]
[0.995007488215134]
[0.9951082060448742]
[0.9951967711095394]
[0.9952631745004296]
[0.9953207152102528]
[0.9953787419153239]
[0.9954208510927596]
[0.995460846751036]
[0.9954831498594904]
[0.9955192357132558]
[0.9955442285929937]
[0.9955632609817072]
[0.9955851733013501]
[0.9955995895097252]
[0.9956192423730709]
[0.9956243493892255]
[0.9956338412572852]


...

[-0.006091578094999495]
[0.6583171833505672]
[0.9509681033367015]
[0.9791042781768292]
[0.9862042207233453]
[0.9892957847992604]
[0.9909560233963975]
[0.9919914915842226]
[0.992667702799798]
[0.9931406303392508]
[0.9934885416187489]
[0.9937596429647158]
[0.9939596664415722]
[0.9941245561688961]
[0.9942574496424251]
[0.9943589287052489]
[0.9944591968604294]
[0.9945326793102283]
[0.994602140735408]
[0.9946581922555778]
[0.9947006349847274]
[0.9947479253090865]
[0.9947786342627255]
[0.9948121885025286]
[0.9948307442291975]
[0.9948605482193041]
[0.9948799212796787]
[0.9949076280189946]
[0.994913767323652]
[0.9949263675329133]
[0.9949275147259898]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.008895915738560634]
[0.6628670364405108]
[0.9468257125593244]
[0.9759528059276305]
[0.9832560615292311]
[0.9863718660713922]
[0.988018581857829]
[0.9890337197895767]
[0.9897085751925963]
[0.9901952489002741]
[0.9905490001642561]
[0.9908241357220989]
[0.9910350766697861]
[0.9912088371343593]
[0.9913528198853319]
[0.9914690158768356]
[0.9915691108850673]
[0.9916446221297163]
[0.991707848072043]
[0.9917752913945385]
[0.9918255601642315]
[0.991855633323619]
[0.9918982366817602]
[0.9919327817783459]
[0.9919512861499853]
[0.9919709143284645]
[0.99200279383779]
[0.9920274524943701]
[0.9920330263625803]
[0.9920493510919528]
[0.9920590056110914]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.011227556598028654]
[0.6779387392268212]
[0.9550752922578933]
[0.980780469689542]
[0.9872110468100103]
[0.9901180155445285]
[0.9917168963422542]
[0.9926955811369991]
[0.9933557826554614]
[0.9938072163637522]
[0.9941493857656162]
[0.9943985710004801]
[0.9946024441620109]
[0.9947562416159683]
[0.9948851259583301]
[0.9949822299348008]
[0.9950753572061944]
[0.9951524026593814]
[0.9952124042248959]
[0.995258845638478]
[0.9953140246853069]
[0.9953397936566596]
[0.995382573546153]
[0.995413574863602]
[0.9954399590033189]
[0.9954646971623761]
[0.9954893273528748]
[0.9955017562034596]
[0.9955081100842715]
[0.9955253381001818]
[0.9955216372020059]


In [26]:
path = 'data/chaos_exp_data_20251022/baseline_L=50_20251019/'
n = 4
d = 2**n
L = 50

spam_list = []
model_list = []

for i in tqdm(range(4)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)



model_saver(spam_list, f'models/integrable_spam_{n}_L=50.model')
model_saver(model_list, f'models/integrable_baseline_model_{n}_L=50.model')

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.005146719934517652]
[0.8231378748789391]
[0.9388583198639271]
[0.961922915901274]
[0.9704448241137595]
[0.9747625040144456]
[0.9773127425422033]
[0.978975229335343]
[0.9801686365520318]
[0.9810383321680186]
[0.9816952807823622]
[0.9822257488554403]
[0.9826275141292673]
[0.9829900029433822]
[0.9832490986306874]
[0.9834835047477607]
[0.9836777982799649]
[0.9838715658187464]
[0.9840265010794663]
[0.9841501745876841]
[0.9842282727640661]
[0.9843348234932049]
[0.9844413877589553]
[0.9845009761918865]
[0.9845800798825132]
[0.9846396681415411]
[0.9846997429804828]
[0.9847322533106914]
[0.9848003577419023]
[0.9848282611810536]
[0.9848484319905177]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.0193585686167701]
[0.8242205096445643]
[0.943118095414351]
[0.9654429928556167]
[0.9737183405605542]
[0.9779512445752501]
[0.9803968604711212]
[0.9819693419859016]
[0.9830343460123993]
[0.9838317642960864]
[0.9844168209085241]
[0.9848817746860492]
[0.9852546234478868]
[0.9855564962150822]
[0.9858088916350938]
[0.9860129997350034]
[0.9861773929186802]
[0.9863380295779411]
[0.9864590478391071]
[0.9865682391828717]
[0.9866634513263665]
[0.9867456511696607]
[0.9868218954688437]
[0.9868974154633848]
[0.9869619205681632]
[0.9870202519603719]
[0.9870549717868137]
[0.9870855965419392]
[0.9871258558285411]
[0.9871753955060489]
[0.9872013384250333]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.009506280535984235]
[0.8172638658934887]
[0.9343323975134966]
[0.9587877960500168]
[0.9680904859687842]
[0.9727786370309293]
[0.9755368939162764]
[0.9773224947876111]
[0.9785680443406544]
[0.9794681586178451]
[0.9801596341425848]
[0.9807055203413103]
[0.9811092086232737]
[0.9814722212640586]
[0.9817521286621183]
[0.982007881734063]
[0.9822116400675488]
[0.9823853983981807]
[0.9825387731307055]
[0.9826635878137094]
[0.9827653152520107]
[0.9828928820983598]
[0.9829686562044543]
[0.9830491597516148]
[0.9831105490824854]
[0.9831724790057375]
[0.9832233183346556]
[0.9832791318879236]
[0.9833212855871907]
[0.983376253110751]
[0.9833976361541046]


FileNotFoundError: [Errno 2] No such file or directory: 'data/chaos_exp_data_20251022/baseline_L=50_20251019/seed_45.pkl'

In [21]:
path = 'data/chaos_exp_data_20251022/intermediate_L=5_20251019/'
n = 4
d = 2**n
L = 5

spam_list = []
model_list = []

for i in tqdm(range(5)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)



model_saver(spam_list, f'models/integrable_spam_{n}_L=5.model')
model_saver(model_list, f'models/integrable_intermediate_model_{n}_L=5.model')

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.0007326438087269516]
[0.539378144116441]
[0.9290923123128579]
[0.9876320203603737]
[0.9935517928681259]
[0.9953320572263575]
[0.996183632907977]
[0.9966598016270555]
[0.996959010165015]
[0.9971624643463194]
[0.9973084293812334]
[0.9974146417719484]
[0.9974955026671468]
[0.9975654851333567]
[0.9976147663085705]
[0.9976591984099696]
[0.9976912974428296]
[0.9977231383163188]
[0.9977407922983872]
[0.9977641239085928]
[0.9977792194137409]
[0.9977964863121378]
[0.9978028602192656]
[0.9978139720944854]
[0.9978302006043422]
[0.9978334486793901]
[0.9978392241849945]
[0.9978491370811023]
[0.9978486611258431]
[0.9978556922792013]
[0.9978608754131124]


  0%|          | 0/3000 [00:00<?, ?it/s]

[0.0022384952141337733]
[0.5845985617386618]
[0.9226291101993599]
[0.9888283643054621]
[0.9954119761696819]
[0.99659275180367]
[0.9971574710377387]
[0.9975221602866201]
[0.9977680754235745]
[0.9979423880414862]
[0.998068645281823]
[0.9981618915772862]
[0.9982372967564914]
[0.9982928832035532]
[0.9983393121370688]
[0.9983761551602153]
[0.998407491183628]
[0.9984351184344166]
[0.9984527865176936]
[0.9984719824514141]
[0.9984871593982648]
[0.998505922926769]
[0.9985140375943531]
[0.9985208095329707]
[0.9985301735111872]
[0.9985402872265275]
[0.99854081612974]
[0.9985503027357778]
[0.998549248358873]
[0.998557212281978]
[0.9985571611802456]


...

[0.003169676102638852]
[0.5459696697717538]
[0.9344153298491207]
[0.9908527120378083]
[0.9952424861947906]
[0.9963085553440352]
[0.9968482303280006]
[0.997175578499239]
[0.9973929376243817]
[0.9975403174388376]
[0.9976502201187236]
[0.9977351127044043]
[0.9977978333926015]
[0.997847913055831]
[0.9978923767404443]
[0.9979283001488829]
[0.9979546670580683]
[0.997979891270206]
[0.9980057398970069]
[0.9980225764606012]
[0.9980386958644438]
[0.9980508655543904]
[0.9980617786299197]
[0.9980659148915657]
[0.9980796990123779]
[0.9980804601993587]
[0.9980966010855544]
[0.9980970351848827]
[0.998101548233832]
[0.9981044190245089]
[0.9981116207210311]


  0%|          | 0/3000 [00:00<?, ?it/s]

[0.0014264492403092133]
[0.5631000032109018]
[0.928376090916814]
[0.9883322134465687]
[0.9941439293972942]
[0.9956815930467747]
[0.9964176702318339]
[0.9968314256095325]
[0.9970943562309647]
[0.9972657211403239]
[0.9973901124830497]
[0.9974831312363605]
[0.9975553118880479]
[0.9976116445990532]
[0.9976627524553989]
[0.9976953630863056]
[0.9977276845491946]
[0.9977529526241756]
[0.9977802834343065]
[0.9978028165280015]
[0.9978101904366484]
[0.9978267646711917]
[0.9978397492047294]
[0.9978528737145064]
[0.9978556993346698]
[0.9978657470783192]
[0.997872674711599]
[0.9978767183966007]
[0.9978833891240915]
[0.9978856186718281]
[0.9978907258040121]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.002496407438229431]
[0.5172767644777083]
[0.9274106917193665]
[0.9895954148064834]
[0.9942375814927202]
[0.9954666595091087]
[0.9961056642643615]
[0.9964962405754423]
[0.9967598672775407]
[0.9969454102334532]
[0.9970816605314939]
[0.9971842781978127]
[0.9972689101670488]
[0.997329819798618]
[0.9973854429265503]
[0.9974247423635193]
[0.9974607190482313]
[0.9974875828339705]
[0.9975140070311757]
[0.9975383416539892]
[0.9975553360260091]
[0.9975738571424191]
[0.9975773681344685]
[0.9975938539568299]
[0.9976009786022016]
[0.9976163785427083]
[0.9976162592595318]
[0.9976213813481689]
[0.9976153396481435]
[0.9976303897923451]
[0.9976424510836166]


In [28]:
path = 'data/chaos_exp_data_20251022/intermediate_L=20_20251019/'
n = 4
d = 2**n
L = 20

spam_list = []
model_list = []

for i in tqdm(range(4)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)



model_saver(spam_list, f'models/integrable_spam_{n}_L=20.model')
model_saver(model_list, f'models/integrable_intermediate_model_{n}_L=20.model')

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.002771373214449868]
[0.7163711808382259]
[0.9492554342117802]
[0.9777898949086604]
[0.9848717645779076]
[0.987903851677255]
[0.9895786805769161]
[0.990641311704324]
[0.991363567330671]
[0.9918840201593186]
[0.9922705583423356]
[0.9925671013948921]
[0.9928057715525496]
[0.9930057277487239]
[0.993162860857849]
[0.9932921050390683]
[0.9933993575722867]
[0.9935074081976832]
[0.9935843820965803]
[0.9936540087401637]
[0.9937091985315408]
[0.9937553334510797]
[0.9937987861453975]
[0.9938407210843545]
[0.9938693694528872]
[0.9939011709143043]
[0.9939366245013188]
[0.9939593567839513]
[0.9939748540945615]
[0.994005474709373]
[0.9939993056187522]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.001270982648019503]
[0.7043968524784254]
[0.9589660087061508]
[0.9807305426940651]
[0.9867191849282133]
[0.989486068904214]
[0.9910408288084359]
[0.9920069770889857]
[0.9926633813507499]
[0.9931222142658225]
[0.9934661186457358]
[0.9937331706544005]
[0.9939333735770721]
[0.9941024332107518]
[0.9942383200084836]
[0.9943483538274508]
[0.9944443310613088]
[0.9945304778420917]
[0.9945937543402028]
[0.9946498982434594]
[0.9946982611811197]
[0.9947486683085636]
[0.9947746795985044]
[0.9948087357142569]
[0.9948413379852722]
[0.9948647541342792]
[0.9948869254662653]
[0.9948951606685387]
[0.9949233643928007]
[0.9949416405639748]
[0.9949567422517203]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.00861227930084385]
[0.6879921327558189]
[0.9444409961975878]
[0.9752736007967571]
[0.9831227659044071]
[0.9865081758467017]
[0.9883344005507021]
[0.9894777474317218]
[0.9902542845225932]
[0.9908169514205536]
[0.9912209887021844]
[0.9915649285963624]
[0.9918162121463221]
[0.9920233526979254]
[0.9921940261098035]
[0.99233916510005]
[0.9924519023642453]
[0.9925458743929694]
[0.992642764440298]
[0.9927164925127804]
[0.9927737766402268]
[0.9928367303594902]
[0.9928778185618611]
[0.9929159877110901]
[0.9929455683767379]
[0.9929820383892274]
[0.9930092067294526]
[0.993032158063155]
[0.9930532742493607]
[0.9930790385356132]
[0.9930643152601385]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.009016434790493522]
[0.712184128072584]
[0.946193767954834]
[0.9769283197143201]
[0.9848848037915175]
[0.9881711365557945]
[0.9899046308995781]
[0.9909744509493489]
[0.9916825380791731]
[0.992194783792398]
[0.9925710017097213]
[0.992864130296751]
[0.9930900731171025]
[0.9932833009262472]
[0.9934385316232507]
[0.9935671482147439]
[0.9936713696870161]
[0.9937650260371422]
[0.9938443143589158]
[0.9939087267226635]
[0.9939751963615008]
[0.9940157231304496]
[0.9940569927985256]
[0.9940948383075847]
[0.9941308844304719]
[0.9941671495135117]
[0.9941855190920473]
[0.9942044581827869]
[0.9942292248982681]
[0.9942452116675807]
[0.9942637439143687]


In [29]:
path = 'data/chaos_exp_data_20251022/intermediate_L=50_20251019/'
n = 4
d = 2**n
L = 50

spam_list = []
model_list = []

for i in tqdm(range(3)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)



model_saver(spam_list, f'models/integrable_spam_{n}_L=50.model')
model_saver(model_list, f'models/integrable_intermediate_model_{n}_L=50.model')

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.006492079273976659]
[0.859966545009881]
[0.9407311317674601]
[0.9615443826967136]
[0.9693993171543127]
[0.973528901478241]
[0.9760623607717376]
[0.9777477013194538]
[0.9789791710187367]
[0.979905644077524]
[0.9806213938614399]
[0.981173957597237]
[0.981629979444522]
[0.9820043140286124]
[0.982296410351021]
[0.982542428424619]
[0.9827579694310241]
[0.9829637059635619]
[0.9831286414731357]
[0.9832882176409838]
[0.9833962425284142]
[0.9835078575412473]
[0.9835731603772315]
[0.9836729149477224]
[0.9837600886822981]
[0.9838045146855536]
[0.9838555805129618]
[0.9839611799373508]
[0.9839741054777233]
[0.9840202934405412]
[0.9840759802315985]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.011655291175060034]
[0.8501773174764878]
[0.9300543270864183]
[0.9502569688864986]
[0.9582403863957911]
[0.9625608726952158]
[0.9653003796852716]
[0.9671640002924049]
[0.9684700789570123]
[0.9695208277632076]
[0.9703057807093112]
[0.9709204900955145]
[0.9714169934411947]
[0.9718467424628058]
[0.9721893554592488]
[0.9724669947909841]
[0.972723080626467]
[0.9729512483312704]
[0.9731142069398704]
[0.9732619565038716]
[0.9733652432975891]
[0.9735320199879727]
[0.9736022033981226]
[0.9737148776355682]
[0.9738265669034686]
[0.9739152770625358]
[0.9739202269719878]
[0.9739503420511012]
[0.9740748872315671]
[0.9741247102084147]
[0.974146019121709]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.018092292290897483]
[0.836468049093752]
[0.9375224964400006]
[0.961070391402892]
[0.9701860954822742]
[0.9748170189168952]
[0.9775417009236946]
[0.9793336719829252]
[0.9805490390946557]
[0.9814483464056314]
[0.98212491796362]
[0.9826791782019734]
[0.9831019712073403]
[0.9834736617156858]
[0.9837496149625395]
[0.9839944354286141]
[0.9841993403445324]
[0.9843666292761704]
[0.9845472424425258]
[0.9846665331114889]
[0.984754066778646]
[0.9848992985991508]
[0.9849675153820608]
[0.9850609586062438]
[0.9851171977705174]
[0.985158708547476]
[0.9852323223395525]
[0.9853108450275391]
[0.9853350694182567]
[0.9853782622240677]
[0.9854171910374561]


## Non-integrable

In [37]:
def load_data(filename, n, seed, L):
    with open(filename, 'rb') as f:
        data = pkl.load(f)


    data = marginalize_counts(data, 0)

    targets = counts_to_probs(data)
    targets_spam = targets[:6**n]
    targets_map = targets[6**n:]

    np.random.seed(seed)
    
    circuit_target = nonintegrable_circuit(n+1, L)
    unitary = Operator(circuit_target).data

    inputs_spam, _ = generate_pauliInput_circuits(n)
        
    inputs_map, circuit_list_map = (
                generate_pauli_circuits(n, None, N=5000-6**n)
        )
    
    return inputs_spam, targets_spam, inputs_map, targets_map, unitary

In [38]:
path = 'data/chaos_exp_data_20251022/nonintegrable_L=5_20251019/'
n = 4
d = 2**n
L = 5

spam_list = []
model_list = []

for i in tqdm(range(3)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)

model_saver(model_list, f'models/nonintegrable_model_{n}_L=5.model')

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.005527530989721363]
[0.6354423318937736]
[0.9557608376252111]
[0.9872591152256905]
[0.9936290486553595]
[0.9953039910846831]
[0.9959560898524288]
[0.9963236970249233]
[0.9965662648629895]
[0.9967408989518615]
[0.9968760003824975]
[0.9969780922107733]
[0.9970605523625056]
[0.9971312891669132]
[0.9971828482692896]
[0.9972304951445001]
[0.9972705253558014]
[0.9973071423061631]
[0.9973323487862676]
[0.9973590037689973]
[0.9973775296642106]
[0.9973968137684971]
[0.9974175126060244]
[0.9974254142545177]
[0.9974411169505839]
[0.9974512118411236]
[0.9974574072542475]
[0.997465565529928]
[0.9974713975303385]
[0.9974879773311655]
[0.9974870505574813]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.005786518644112615]
[0.6282146292206616]
[0.9635687362442746]
[0.989962086251552]
[0.9940909135628743]
[0.9957316200424794]
[0.9964989075694267]
[0.996907070152001]
[0.9971598086526436]
[0.9973303618708756]
[0.9974575630724746]
[0.9975545752673347]
[0.9976325483440298]
[0.9976957853903965]
[0.9977454624102678]
[0.9977898633370846]
[0.997826384381918]
[0.9978568710355571]
[0.9978841861572778]
[0.9979079265759919]
[0.9979259644903145]
[0.9979444355609984]
[0.9979600040272713]
[0.9979740815139048]
[0.9979825532912779]
[0.9979959191682598]
[0.9979997803240744]
[0.9980085906110868]
[0.998016853769193]
[0.9980231375146582]
[0.998024290986659]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.002474556704208597]
[0.6394258088982343]
[0.9471790197015315]
[0.9875571525531667]
[0.9938366225364891]
[0.9952371933700748]
[0.9957789840715144]
[0.9960927736862252]
[0.9963047085939173]
[0.9964624830544062]
[0.9965811673646439]
[0.9966809683874696]
[0.9967582665165756]
[0.9968241906416696]
[0.9968823254826721]
[0.9969266934648575]
[0.9969677681540772]
[0.9970026689146876]
[0.9970304175726454]
[0.9970577339332971]
[0.9970781566221789]
[0.9970986809177351]
[0.9971147311397205]
[0.997124744698941]
[0.9971351951102357]
[0.9971519042004704]
[0.9971587178076486]
[0.9971703262311893]
[0.9971740251658288]
[0.9971800029231764]
[0.9971797970970553]


# Integrable

In [36]:
def load_data(filename, n, seed, L):
    with open(filename, 'rb') as f:
        data = pkl.load(f)


    data = marginalize_counts(data, 0)

    targets = counts_to_probs(data)
    targets_spam = targets[:6**n]
    targets_map = targets[6**n:]

    np.random.seed(seed)
    
    circuit_target = integrable_circuit(n+1, L)
    unitary = Operator(circuit_target).data

    inputs_spam, _ = generate_pauliInput_circuits(n)
        
    inputs_map, circuit_list_map = (
                generate_pauli_circuits(n, None, N=5000-6**n)
        )
    
    return inputs_spam, targets_spam, inputs_map, targets_map, unitary

path = 'data/chaos_exp_data_20251022/integrable_L=5_20251019/'
n = 4
d = 2**n
L = 5

spam_list = []
model_list = []

for i in tqdm(range(2)):
    seed = 52 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)

model_saver(model_list, f'models/integrable_model_{n}_L=5.model')

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

[-6.152765006373073e-06]
[0.6144343423659369]
[0.9357217321718864]
[0.988038814094913]
[0.9945704699279216]
[0.9961361428125829]
[0.9968004853315725]
[0.9971692792954453]
[0.9973984384100013]
[0.9975587667168164]
[0.9976748620353046]
[0.997763591589802]
[0.997833101175043]
[0.9978855785821519]
[0.9979361646407323]
[0.9979708877866564]
[0.9980041102577386]
[0.9980278387652215]
[0.998054299171926]
[0.9980746190464183]
[0.9980912939001703]
[0.9981039586681513]
[0.9981188079315243]
[0.9981254955992797]
[0.9981376108775684]
[0.9981477577756487]
[0.9981562006187358]
[0.998163007690237]
[0.9981697403154994]
[0.9981749315363572]
[0.9981733471546343]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.0012793622763824786]
[0.6117998095668248]
[0.9284802239362066]
[0.9861583906814008]
[0.9938963243090722]
[0.9956911885187752]
[0.9964073634956336]
[0.9967823127940512]
[0.9970133558224148]
[0.9971739392857228]
[0.9972895298511265]
[0.9973749144081151]
[0.9974423619839635]
[0.9974999978886021]
[0.9975436152053878]
[0.997581634302939]
[0.9976106867383137]
[0.9976354016712852]
[0.9976571351978236]
[0.9976720220655607]
[0.9976938471358555]
[0.9977101595892435]
[0.9977201619767596]
[0.9977292483194924]
[0.9977418109976955]
[0.9977435685555186]
[0.9977563112175093]
[0.9977582619206227]
[0.9977654414545337]
[0.9977706811187558]
[0.9977773101649908]
